## Receive the parameters

In [0]:
from pyspark.sql.functions import *
from datetime import date

dbutils.widgets.text("selected_date", "")
selected_date = dbutils.widgets.get("selected_date")

if not selected_date:
    raise ValueError("Missing selected_date parameter")

selected_date = date.fromisoformat(selected_date)

dbutils.widgets.text("file_path", "")
file_path = dbutils.widgets.get("file_path")

if not file_path:
    raise ValueError("Missing file_path parameter")

dbutils.widgets.text("row_count", "")
row_count = dbutils.widgets.get("row_count")

if not row_count:
    raise ValueError("Missing row_count parameter")

row_count = int(row_count)

## Combine the values into a dataframe

In [0]:
data = [(selected_date, file_path, row_count)]

columns = [
    "transaction_date",
    "file_path",
    "row_count"
]

df_progress = spark.createDataFrame(data, columns)

# A temporary view allows us to query a dataframe using SQL
df_progress.createOrReplaceTempView("publication_update")


## Update transaction table

In [0]:
%sql
MERGE INTO retail_project.external_app_schema.simulator_progress AS target
USING publication_update AS source

ON target.transaction_date = source.transaction_date

WHEN MATCHED THEN
    UPDATE SET
        target.file_path = source.file_path,
        target.row_count = source.row_count

WHEN NOT MATCHED THEN
    INSERT (
        transaction_date,
        file_path,
        row_count,
        published_at
    )
    VALUES (
        source.transaction_date,
        source.file_path,
        source.row_count,
        current_timestamp()
    );